# Sesión 2 — ¿Poisson, Gamma o Tweedie?
## Módulo 4 · Tema 2: GLM con Python
### Diplomado: Machine Learning en Seguros · FC UNAM
### 26 de agosto de 2026  ·  3 horas

---

> **Premisa:** ya modelaste la **frecuencia**. Hoy completas la prima pura con la
> **severidad** (Gamma), aprendes a tratar los **siniestros grandes**, conoces el
> atajo agregado (**Tweedie**), y —lo central— aprendes a **elegir el mejor modelo**
> con AIC, BIC, pseudo R² y **selección automatizada (stepwise)**.
>
> Al final, este notebook **guarda los tres mejores modelos** (frecuencia, severidad,
> agregado) en `modelos/` — son los que usaremos en la Sesión 3.

---

**Prerequisito:** `datos/datos.pkl`. Ambiente `diplomado` con `statsmodels`, `scikit-learn`,
`pandas`, `numpy`, `scipy`, `matplotlib`, `plotly`.
Soluciones en `m4t2_s2_soluciones.py`.

## Ruta de la sesión — seis bloques

| # | Bloque | Idea central |
|---|--------|--------------|
| 1 | Severidad con Gamma | El monto por siniestro · CV constante · por qué Lognormal no es GLM |
| 2 | Large losses | El 1% de siniestros que distorsiona todo · capping, split, reaseguro |
| 3 | Tweedie | La prima pura de un jalón · compound Poisson-Gamma |
| 4 | Comparar modelos | AIC · BIC · pseudo R² (y por qué en seguros es bajo) |
| 5 | Selección automatizada | Stepwise forward/backward · sus trampas · LASSO como sucesor |
| 6 | Los mejores modelos + Python vs R | Generamos y **guardamos** freq/sev/agregado para la Sesión 3 |

In [2]:
%pip install statsmodels

  Using cached statsmodels-0.14.6-cp311-cp311-macosx_10_9_x86_64.whl.metadata (9.5 kB)
  Using cached patsy-1.0.2-py2.py3-none-any.whl.metadata (3.6 kB)
Using cached statsmodels-0.14.6-cp311-cp311-macosx_10_9_x86_64.whl (10.1 MB)
Using cached patsy-1.0.2-py2.py3-none-any.whl (233 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [statsmodels] [statsmodels]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import numpy as np, pandas as pd, os, json
import matplotlib.pyplot as plt
import statsmodels.api as sm, statsmodels.formula.api as smf
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

AZUL='#0D7A8A'; AMBAR='#F4A261'; GRIS='#8FA3B1'; MORADO='#B07CC6'; VERDE='#2ECC71'
plt.rcParams.update({'figure.figsize':(10,5),'axes.grid':True,'grid.alpha':0.25,'axes.axisbelow':True})
pd.set_option('display.float_format','{:,.4f}'.format); pd.set_option('display.max_columns',25)

datos = pd.read_pickle('datos/datos.pkl')

# Bandings (mismos de la Sesión 1)
BANDINGS = {'edad_conductor':[17,30,35,45,50,55,60,95],
            'antiguedad_vehiculo':[-1,1,2,3,4,5,10,15,50],
            'potencia':[9,40,50,60,70,250],
            'nivel_bonus':[-1,0,1,2,5,10,25]}
for c,ct in BANDINGS.items():
    datos[c+'_cat'] = pd.cut(datos[c], bins=ct).astype(str)

# Para severidad: SOLO pólizas con siniestro (N > 0)
sev = datos[datos['num_siniestros'] > 0].copy()
freq_ref = datos['num_siniestros'].sum()/datos['exposicion'].sum()
sev_ref  = np.average(sev['severidad'], weights=sev['num_siniestros'])
print(f'Cartera: {len(datos):,} pólizas · con siniestro: {len(sev):,} ({len(sev)/len(datos):.1%})')
print(f'Frecuencia ref: {freq_ref:.4%}  ·  Severidad ref: ${sev_ref:,.0f}')

Cartera: 163,212 pólizas · con siniestro: 18,276 (11.2%)
Frecuencia ref: 13.9206%  ·  Severidad ref: $1,309


---
# Bloque 1 · Severidad con Gamma

La prima pura es $\text{Frecuencia}\times\text{Severidad}$. Ya tienes la frecuencia;
la **severidad** es el **monto promedio por siniestro**, y cambia todo respecto a la
frecuencia: la respuesta es un **monto positivo** ($Y>0$), no un conteo, y **solo se
modela sobre pólizas con siniestro** ($N>0$).

La distribución natural es la **Gamma con liga log**. Su propiedad clave es el
**coeficiente de variación constante**: para $Y\sim\text{Gamma}(\alpha,\beta)$,
$$\text{CV}=\frac{\sigma}{\mu}=\frac{1}{\sqrt{\alpha}}=\text{constante}.$$
Actuarialmente tiene sentido: un siniestro de \$100,000 tiene más variabilidad **en pesos**
que uno de \$1,000, pero la variabilidad **relativa** (como % del monto) es parecida.

In [5]:
# ── EDA: severidad empírica por variable, ponderada por número de siniestros ─
def severidad_empirica(df, variable):
    g = df.groupby(variable, observed=True)
    out = g.apply(lambda x: pd.Series({
        'nclaims': x['num_siniestros'].sum(),
        'sev':  np.average(x['severidad'], weights=x['num_siniestros']),
        'std':  np.sqrt(np.average((x['severidad']-np.average(x['severidad'],weights=x['num_siniestros']))**2,
                                   weights=x['num_siniestros'])),
    }), include_groups=False).reset_index()
    out['CV'] = out['std']/out['sev']
    return out

sev_cob = severidad_empirica(sev, 'cobertura')
print('Severidad y CV por cobertura:')
print(sev_cob.round(3).to_string(index=False))
print('\nSi el CV es parecido entre grupos, la Gamma es apropiada.')

Severidad y CV por cobertura:
cobertura     nclaims        sev        std     CV
   Amplia  2,675.0000 1,588.9170 3,987.2590 2.5090
 Limitada  5,322.0000 1,061.4930 2,876.7050 2.7100
       RC 12,218.0000 1,355.8150 3,510.3630 2.5890

Si el CV es parecido entre grupos, la Gamma es apropiada.


In [4]:
# ── Las dos formas de ajustar Gamma (el paralelo de offset/weights en frecuencia)
# Forma 1: respuesta = severidad promedio (average), peso = número de siniestros
g_avg = smf.glm('severidad ~ C(cobertura)', sev,
                family=sm.families.Gamma(sm.families.links.Log()),
                freq_weights=sev['num_siniestros']).fit()
# Forma 2: respuesta = monto total, offset = log(nclaims)
g_amt = smf.glm('monto_total ~ C(cobertura)', sev,
                family=sm.families.Gamma(sm.families.links.Log()),
                offset=np.log(sev['num_siniestros']),
                freq_weights=sev['num_siniestros']).fit()
comp = pd.DataFrame({'forma1_average':g_avg.params, 'forma2_amount':g_amt.params})
comp['dif'] = (comp['forma1_average']-comp['forma2_amount']).abs()
print(comp.to_string())
print(f'\nDiferencia máxima: {comp["dif"].max():.2e}  →  mismas formas, mismo modelo.')
print(f'CV implícito del modelo (√escala): {np.sqrt(g_avg.scale):.3f}')

                          forma1_average  forma2_amount    dif
Intercept                         7.3708         7.3708 0.0000
C(cobertura)[T.Limitada]         -0.4034        -0.4034 0.0000
C(cobertura)[T.RC]               -0.1586        -0.1586 0.0000

Diferencia máxima: 1.50e-15  →  mismas formas, mismo modelo.
CV implícito del modelo (√escala): 2.611


### ¿Por qué Lognormal NO es un GLM?

Es tentador modelar $\log(Y)$ con una regresión lineal normal — pero eso **no** es un GLM
sobre $Y$. La razón es formal: la familia exponencial **natural** (NEF) exige que el
estadístico suficiente sea $T(y)=y$. Para la Lognormal, el estadístico suficiente es
$T(y)=\log(y)$, así que **no pertenece a la NEF** y no puede usarse directamente en `glm()`.

Consecuencia práctica: una regresión sobre $\log(Y)$ modela $\mathbb{E}[\log Y]$, **no**
$\mathbb{E}[Y]$, y volver a la escala original exige un **factor de corrección por sesgo**
$\exp(\hat\sigma^2/2)$ que introduce error. La **Gamma** con liga log modela $\mathbb{E}[Y]$
directamente, sin corrección — por eso es la opción recomendada para severidad.

### 📝 Ejercicio 1 — El CV constante en otra variable (8 min)

- **1a.** Calcula la severidad empírica y el CV por `edad_conductor_cat` con `severidad_empirica`.
- **1b.** ¿El CV se mantiene relativamente constante entre grupos? ¿Justifica usar Gamma?
- **1c.** Ajusta `severidad ~ C(edad_conductor_cat)` (Gamma, log, weights) y reporta √escala.

*Solución en `solucion_ejercicio1()`.*

In [6]:
# Tu código aquí:

# ============================================================
# EJERCICIO 1 — CV constante por edad del conductor
# ============================================================

# 1a. Severidad empírica y CV por edad_conductor_cat
sev_edad = severidad_empirica(
    sev,
    'edad_conductor_cat'
)

print("Severidad empírica y CV por categoría de edad:")
print(
    sev_edad
    .round(3)
    .to_string(index=False)
)

print()


# ============================================================
# 1b. Revisar si el CV es relativamente constante
# ============================================================

cv_promedio = sev_edad['CV'].mean()
cv_min = sev_edad['CV'].min()
cv_max = sev_edad['CV'].max()

print("Resumen del CV entre categorías:")
print(f"CV promedio: {cv_promedio:.3f}")
print(f"CV mínimo:   {cv_min:.3f}")
print(f"CV máximo:   {cv_max:.3f}")
print(f"Rango CV:    {cv_max - cv_min:.3f}")

print()

# Regla descriptiva sencilla
variacion_relativa_cv = (
    (cv_max - cv_min) / cv_promedio
)

if variacion_relativa_cv < 0.25:
    print(
        "Interpretación: el CV es relativamente estable entre "
        "los grupos de edad."
    )
    print(
        "Esto apoya el uso de una distribución Gamma para "
        "modelar la severidad."
    )
else:
    print(
        "Interpretación: el CV presenta diferencias importantes "
        "entre los grupos de edad."
    )
    print(
        "La hipótesis de CV constante de la Gamma debe tomarse "
        "con cautela."
    )


# ============================================================
# 1c. Modelo Gamma con liga log
# ============================================================

g_edad = smf.glm(
    formula='severidad ~ C(edad_conductor_cat)',
    data=sev,
    family=sm.families.Gamma(
        link=sm.families.links.Log()
    ),
    freq_weights=sev['num_siniestros']
).fit()

print()
print("=" * 60)
print("MODELO GAMMA — Severidad por edad del conductor")
print("=" * 60)

print(
    g_edad.summary()
)

# CV implícito del modelo Gamma
cv_modelo = np.sqrt(
    g_edad.scale
)

print()
print(
    f"CV implícito del modelo (√escala): "
    f"{cv_modelo:.3f}"
)


Severidad empírica y CV por categoría de edad:
edad_conductor_cat    nclaims        sev        std     CV
          (17, 30] 4,461.0000 1,478.0680 3,960.7570 2.6800
          (30, 35] 2,452.0000 1,361.6300 4,161.1640 3.0560
          (35, 45] 4,770.0000 1,231.8560 3,061.3910 2.4850
          (45, 50] 2,275.0000 1,337.2790 3,432.1580 2.5670
          (50, 55] 1,790.0000 1,144.4740 2,518.5830 2.2010
          (55, 60] 1,283.0000 1,109.3100 1,972.9910 1.7790
          (60, 95] 3,184.0000 1,301.0290 3,415.0560 2.6250

Resumen del CV entre categorías:
CV promedio: 2.485
CV mínimo:   1.779
CV máximo:   3.056
Rango CV:    1.277

Interpretación: el CV presenta diferencias importantes entre los grupos de edad.
La hipótesis de CV constante de la Gamma debe tomarse con cautela.

MODELO GAMMA — Severidad por edad del conductor
                 Generalized Linear Model Regression Results                  
Dep. Variable:              severidad   No. Observations:                18276
Model:         

---
# Bloque 2 · Large losses — el 1% que distorsiona todo

En severidad, unos pocos siniestros enormes pueden dominar el monto total y desestabilizar
el modelo. Es la regla empírica: **el 1% de los siniestros puede representar el 30% del
monto**. Antes de modelar, hay que **verlo** y **tratarlo**.

In [9]:
# ── La concentración: ¿qué fracción del monto está en los siniestros más grandes?
montos = np.sort(sev['monto_total'].values)[::-1]   # de mayor a menor
cum = np.cumsum(montos)/montos.sum()
for pct in [0.01, 0.05, 0.10]:
    k = int(len(montos)*pct)
    print(f'El {pct:.0%} de siniestros más grandes concentra el {cum[k-1]:.1%} del monto total')

p99 = np.percentile(sev['monto_total'], 99)
print(f'\nPercentil 99 del monto: ${p99:,.0f}  ·  máximo: ${sev["monto_total"].max():,.0f}')

El 1% de siniestros más grandes concentra el 22.1% del monto total
El 5% de siniestros más grandes concentra el 43.1% del monto total
El 10% de siniestros más grandes concentra el 54.5% del monto total

Percentil 99 del monto: $18,335  ·  máximo: $140,032


### Cómo se tratan (opciones, de menos a más agresivo)

- **Capping (truncar) al P99** — limitar los montos por encima del percentil 99. Simple y
  documentable; el exceso se cede a **reaseguro XL**.
- **Modelar por separado** — un modelo para siniestros *attritional* (frecuentes, chicos) y
  otro para *large losses* (raros, grandes).
- **Winsorizar** — reemplazar los extremos por el percentil, menos agresivo que truncar.
- **Distribución robusta** — Gaussiana Inversa ($V(\mu)=\mu^3$) para colas más pesadas.

Veamos el efecto del capping al P99 sobre la severidad estimada:

In [10]:
# ── Efecto del capping al P99 sobre el modelo de severidad ───────────────────
sev['monto_cap'] = np.minimum(sev['monto_total'], p99)
sev['sev_cap']   = sev['monto_cap'] / sev['num_siniestros']

g_sin = smf.glm('severidad ~ C(cobertura)', sev, family=sm.families.Gamma(sm.families.links.Log()),
                freq_weights=sev['num_siniestros']).fit()
g_con = smf.glm('sev_cap ~ C(cobertura)', sev, family=sm.families.Gamma(sm.families.links.Log()),
                freq_weights=sev['num_siniestros']).fit()

print(f'Severidad media SIN capping: ${np.average(sev["severidad"],weights=sev["num_siniestros"]):,.0f}')
print(f'Severidad media CON capping: ${np.average(sev["sev_cap"],weights=sev["num_siniestros"]):,.0f}')
print(f'CV √escala  sin capping: {np.sqrt(g_sin.scale):.3f}')
print(f'CV √escala  con capping: {np.sqrt(g_con.scale):.3f}   (más estable)')
print('\nEl capping baja la severidad media y estabiliza el modelo; el exceso va a reaseguro XL.')
print('Regla CNSF: documentar el tratamiento y el umbral elegido en la nota técnica.')

Severidad media SIN capping: $1,309
Severidad media CON capping: $1,184
CV √escala  sin capping: 2.611
CV √escala  con capping: 1.830   (más estable)

El capping baja la severidad media y estabiliza el modelo; el exceso va a reaseguro XL.
Regla CNSF: documentar el tratamiento y el umbral elegido en la nota técnica.


### 📝 Ejercicio 2 — Sensibilidad al umbral de capping (10 min)

- **2a.** Repite el capping al **P95** y al **P99.5**. ¿Cómo cambia la severidad media?
- **2b.** ¿Qué % del monto total se cede a reaseguro en cada umbral?
- **2c.** ¿Qué umbral defenderías y por qué (estabilidad vs pérdida de información)?

*Solución en `solucion_ejercicio2()`.*

In [11]:
# Tu código aquí:


---
# Bloque 3 · Tweedie — la prima pura de un jalón

Frecuencia × Severidad usa **dos** modelos. **Tweedie** modela la **prima pura**
$\text{PP}_i = \text{monto}_i/\text{exposición}_i$ directamente, en **un solo** modelo.
Su magia: es una **compound Poisson-Gamma** que maneja a la vez la **masa en cero** (pólizas
sin siniestro) y los **valores positivos** (con siniestro). Su función de varianza:
$$V(\mu)=\mu^{p},\qquad 1<p<2.$$
Casos límite: $p=1$ es Poisson (frecuencia pura), $p=2$ es Gamma (severidad pura); en medio
es la mezcla ideal para prima pura.

In [12]:
# ── La prima pura tiene mucha masa en cero y una cola positiva ───────────────
datos['pure_premium'] = datos['monto_total'] / datos['exposicion']
print(f'Pólizas con prima pura = 0 (sin siniestro): {(datos["pure_premium"]==0).mean():.1%}')
print(f'Pólizas con prima pura > 0:                {(datos["pure_premium"]>0).mean():.1%}')

Pólizas con prima pura = 0 (sin siniestro): 88.8%
Pólizas con prima pura > 0:                11.2%


In [13]:
# ── Estimar el parámetro p: barrido con el D² de sklearn (un pseudo R²) ───────
from sklearn.linear_model import TweedieRegressor
feats = ['cobertura','sexo','combustible','edad_conductor_cat','potencia_cat','nivel_bonus_cat']
X = pd.get_dummies(datos[feats], drop_first=True).astype(float).values
y = datos['pure_premium'].values; w = datos['exposicion'].values

perfil = []
for p in [1.2,1.3,1.4,1.5,1.6,1.7,1.8]:
    tw = TweedieRegressor(power=p, alpha=0, link='log', max_iter=400).fit(X, y, sample_weight=w)
    perfil.append({'p':p, 'D2':tw.score(X, y, sample_weight=w)})
perfil = pd.DataFrame(perfil)
p_opt = perfil.loc[perfil['D2'].idxmax(),'p']
print(perfil.round(5).to_string(index=False))
print(f'\np óptimo (máximo D²): {p_opt}')
print('NOTA: sklearn hace un barrido de D²; R (tweedie.profile) estima p por MÁXIMA')
print('VEROSIMILITUD — lo comparamos a fondo en el bloque de Python vs R.')

     p     D2
1.2000 0.0352
1.3000 0.0359
1.4000 0.0354
1.5000 0.0335
1.6000 0.0301
1.7000 0.0252
1.8000 0.0186

p óptimo (máximo D²): 1.3
NOTA: sklearn hace un barrido de D²; R (tweedie.profile) estima p por MÁXIMA
VEROSIMILITUD — lo comparamos a fondo en el bloque de Python vs R.


In [ ]:
# ── Ajustar el GLM Tweedie con el p óptimo y comparar con Freq × Sev ─────────
mod_tweedie = smf.glm(f'pure_premium ~ C(cobertura)+C(sexo)+C(combustible)+C(edad_conductor_cat)+C(potencia_cat)+C(nivel_bonus_cat)',
                      datos, family=sm.families.Tweedie(var_power=p_opt, link=sm.families.links.Log()),
                      var_weights=datos['exposicion']).fit()
pp_obs = (datos['pure_premium']*datos['exposicion']).sum()
pp_tw  = (mod_tweedie.predict(datos)*datos['exposicion']).sum()
print(f'Prima pura total observada: ${pp_obs:,.0f}')
print(f'Prima pura total Tweedie:   ${pp_tw:,.0f}   (ratio {pp_tw/pp_obs:.4f})')
print(f'\nTweedie: {len(mod_tweedie.params)} parámetros, p = {p_opt}')
# a diferencia de los bloques pasados, aqui se tiene que definir el valor de p
# forma parte de una de las cosas que se tienen que calcular


Prima pura total observada: $26,464,970
Prima pura total Tweedie:   $26,464,565   (ratio 1.0000)

Tweedie: 20 parámetros, p = 1.3


---
# Bloque 4 · Comparar modelos — AIC, BIC y pseudo R²

Con varios candidatos ajustados, ¿cuál elegimos? Tres criterios:
$$\text{AIC}=-2\ell+2p,\qquad \text{BIC}=-2\ell+\log(n)\,p,$$
donde $\ell$ es la log-verosimilitud y $p$ el número de parámetros. **Menor es mejor**; BIC
penaliza más la complejidad. Y el **pseudo R²**, que no viene en el `summary`:
$$R^2_{\text{McFadden}}=1-\frac{\ell_{\text{modelo}}}{\ell_{\text{nulo}}},\qquad
R^2_{\text{devianza}}=1-\frac{D_{\text{modelo}}}{D_{\text{nulo}}}.$$

In [15]:
# ── Construimos las funciones de pseudo R² (no vienen en statsmodels) ─────────
def pseudo_r2(modelo, formula_nula, data, **kw):
    m0 = smf.glm(formula_nula, data, family=modelo.family, **kw).fit()
    return {'McFadden': 1 - modelo.llf/m0.llf, 'devianza': 1 - modelo.deviance/m0.deviance}

# Comparación de modelos de frecuencia: Poisson vs Binomial Negativa
f_formula = 'num_siniestros ~ C(cobertura)+C(edad_conductor_cat)+C(nivel_bonus_cat)+C(potencia_cat)'
m_pois = smf.glm(f_formula, datos, family=sm.families.Poisson(), offset=np.log(datos['exposicion'])).fit()
m_nb   = smf.glm(f_formula, datos, family=sm.families.NegativeBinomial(alpha=1.0), offset=np.log(datos['exposicion'])).fit()

tabla = pd.DataFrame({
    'modelo':['Poisson','Binomial Negativa'],
    'AIC':[m_pois.aic, m_nb.aic], 'BIC':[m_pois.bic_llf, m_nb.bic_llf],
    'pseudoR2_McFadden':[pseudo_r2(m_pois,'num_siniestros ~ 1',datos,offset=np.log(datos['exposicion']))['McFadden'],
                         pseudo_r2(m_nb,'num_siniestros ~ 1',datos,offset=np.log(datos['exposicion']))['McFadden']],
})
print(tabla.to_string(index=False, float_format=lambda v: f'{v:,.4f}'))

           modelo          AIC          BIC  pseudoR2_McFadden
          Poisson 125,081.6634 125,261.7139             0.0198
Binomial Negativa 124,925.7126 125,105.7631             0.0177


> **Ojo con el pseudo R² en seguros.** Verás valores **bajos** (0.02–0.05), y está **bien**:
> la frecuencia de siniestros tiene una enorme componente aleatoria irreducible — no es como
> un R² de OLS. Lo que importa es la **comparación relativa** entre modelos y la
> **discriminación** (Gini, lift), que veremos en la Sesión 3, no un número alto.

### 📝 Ejercicio 3 — pseudo R² de la severidad (8 min)

- **3a.** Ajusta un Gamma de severidad con varias variables y calcula su pseudo R² (McFadden
  y devianza) con `pseudo_r2`.
- **3b.** Compáralo contra un modelo con menos variables. ¿Cuál gana en AIC?
- **3c.** ¿El pseudo R² de severidad es mayor o menor que el de frecuencia? ¿Por qué?

*Solución en `solucion_ejercicio3()`.*

In [16]:
# Tu código aquí:


---
# Bloque 5 · Selección automatizada — stepwise

Comparar candidatos a mano no escala. El **stepwise** recorre el espacio de variables
**automáticamente**, guiado por el AIC:

- **Forward** — parte del nulo y en cada paso **agrega** la variable que más baja el AIC.
- **Backward** — parte del saturado y **elimina**.
- **Both directions** — agrega y quita en cada paso.

En la Sesión 1 hicimos este build-up **a mano**; aquí lo **automatizamos**.

In [17]:
# ── Forward stepwise por AIC (el corazón del algoritmo) ──────────────────────
def forward_stepwise(data, respuesta, candidatas, offset=None, family=None, verbose=True):
    family = family or sm.families.Poisson()
    seleccionadas, aic_actual = [], np.inf
    m0 = smf.glm(f'{respuesta} ~ 1', data, family=family, offset=offset).fit()
    aic_actual = m0.aic
    if verbose: print(f'AIC inicial (nulo): {aic_actual:,.1f}')
    while True:
        mejor_var, mejor_aic = None, aic_actual
        for v in candidatas:
            if v in seleccionadas: continue
            f = f'{respuesta} ~ ' + ' + '.join(seleccionadas + [v])
            aic = smf.glm(f, data, family=family, offset=offset).fit().aic
            if aic < mejor_aic: mejor_aic, mejor_var = aic, v
        if mejor_var is None: break
        seleccionadas.append(mejor_var)
        if verbose: print(f'  + {mejor_var:<26} AIC = {mejor_aic:,.1f}  (Δ {mejor_aic-aic_actual:,.1f})')
        aic_actual = mejor_aic
    return seleccionadas

candidatas = ['C(cobertura)','C(sexo)','C(uso)','C(combustible)',
              'C(edad_conductor_cat)','C(antiguedad_vehiculo_cat)','C(potencia_cat)','C(nivel_bonus_cat)']
sel_freq = forward_stepwise(datos, 'num_siniestros', candidatas, offset=np.log(datos['exposicion']))
print('\nVariables seleccionadas (frecuencia):', sel_freq)

AIC inicial (nulo): 127,574.3
  + C(nivel_bonus_cat)         AIC = 125,430.1  (Δ -2,144.2)
  + C(edad_conductor_cat)      AIC = 125,172.5  (Δ -257.6)
  + C(combustible)             AIC = 125,074.3  (Δ -98.2)
  + C(antiguedad_vehiculo_cat) AIC = 124,993.8  (Δ -80.5)
  + C(potencia_cat)            AIC = 124,904.8  (Δ -89.0)
  + C(cobertura)               AIC = 124,893.9  (Δ -11.0)
  + C(uso)                     AIC = 124,890.5  (Δ -3.3)

Variables seleccionadas (frecuencia): ['C(nivel_bonus_cat)', 'C(edad_conductor_cat)', 'C(combustible)', 'C(antiguedad_vehiculo_cat)', 'C(potencia_cat)', 'C(cobertura)', 'C(uso)']


### El matiz con one-hot, y las trampas del stepwise

Si en vez de la variable completa metes sus **dummies** (one-hot), el stepwise selecciona
**categorías individuales** — te dice *qué niveles* importan y sugiere **agrupar** el resto
(p.ej. fusionar zonas de bajo riesgo en una sola). Es más granular, pero más frágil.

Y las **advertencias honestas**:
- Es **greedy**: no garantiza el óptimo global, solo un buen camino local.
- Los **p-values post-selección ya no son válidos**: elegiste las variables mirando los datos,
  así que la inferencia queda sesgada (no reportes esos p-values como si nada).
- **Sobreajusta**: por eso se valida **out-of-sample** (Sesión 3).

El sucesor moderno es la **regularización (LASSO/Ridge/Elastic Net)**: en lugar de agregar/quitar
en saltos, penaliza los coeficientes de forma continua y hace la selección "sola". Es la
antesala del bloque de ML del diplomado.

### 📝 Ejercicio 4 — Stepwise para severidad (12 min)

- **4a.** Adapta `forward_stepwise` para la **severidad** (Gamma, weights=nclaims, sin offset).
- **4b.** ¿Selecciona las mismas variables que en frecuencia? Compara y comenta.
- **4c.** ¿Por qué las variables que importan para *cuántos* siniestros pueden no ser las
  mismas que para *de qué tamaño*?

*Solución en `solucion_ejercicio4()`.*

In [18]:
# Tu código aquí:


---
# Bloque 6 · Los tres mejores modelos — generar y **guardar** para la Sesión 3

Este es el entregable central de hoy: ajustamos y **persistimos** los tres modelos que la
Sesión 3 usará para visualizar efectos, validar, hacer sensibilidad y **simular la tarifa**.

In [19]:
os.makedirs('modelos', exist_ok=True)

# ── (A) Mejor modelo de FRECUENCIA — variables del stepwise, Poisson ─────────
f_final = 'num_siniestros ~ ' + ' + '.join(sel_freq)
modelo_frecuencia = smf.glm(f_final, datos, family=sm.families.Poisson(),
                            offset=np.log(datos['exposicion'])).fit()

# ── (B) Mejor modelo de SEVERIDAD — stepwise sobre Gamma ─────────────────────
sel_sev = forward_stepwise(sev, 'severidad', candidatas,
                           family=sm.families.Gamma(sm.families.links.Log()), verbose=False)
s_final = 'severidad ~ ' + ' + '.join(sel_sev)
modelo_severidad = smf.glm(s_final, sev, family=sm.families.Gamma(sm.families.links.Log()),
                           freq_weights=sev['num_siniestros']).fit()

# ── (C) Mejor modelo AGREGADO — Tweedie con p óptimo ─────────────────────────
modelo_agregado = mod_tweedie   # ajustado en el bloque 3

print('Frecuencia :', f_final)
print('Severidad  :', s_final)
print(f'Agregado   : Tweedie (p={p_opt}), {len(modelo_agregado.params)} parámetros')

Frecuencia : num_siniestros ~ C(nivel_bonus_cat) + C(edad_conductor_cat) + C(combustible) + C(antiguedad_vehiculo_cat) + C(potencia_cat) + C(cobertura) + C(uso)
Severidad  : severidad ~ C(edad_conductor_cat) + C(antiguedad_vehiculo_cat) + C(potencia_cat) + C(nivel_bonus_cat) + C(sexo)
Agregado   : Tweedie (p=1.3), 20 parámetros


In [20]:
# ── Persistir los modelos de forma LIGERA y portable ─────────────────────────
# Serializar los objetos de statsmodels pesa cientos de MB (guardan la matriz de
# diseño). En su lugar guardamos lo que define al modelo: la fórmula, la familia y
# los rating factors. La Sesión 3 los RECONSTRUYE con cargar_modelos.py (refit exacto
# en ~1 s) — más ligero, portable y a prueba de versiones.
def rating_factors(m):
    return pd.DataFrame({'coef':m.params,'RF':np.exp(m.params),
                         'RF_lower':np.exp(m.params-1.96*m.bse),
                         'RF_upper':np.exp(m.params+1.96*m.bse),'p_value':m.pvalues})

tw_formula = ('pure_premium ~ C(cobertura)+C(sexo)+C(combustible)+'
              'C(edad_conductor_cat)+C(potencia_cat)+C(nivel_bonus_cat)')
meta = {
  'frecuencia':{'respuesta':'num_siniestros','formula':f_final,
                'family':'Poisson','offset':'log(exposicion)'},
  'severidad' :{'respuesta':'severidad','formula':s_final,
                'family':'Gamma(log)','weights':'num_siniestros','filtro':'num_siniestros>0'},
  'agregado'  :{'respuesta':'pure_premium','formula':tw_formula,
                'family':'Tweedie','p':float(p_opt),'weights':'exposicion'},
  'bandings':{k:v for k,v in BANDINGS.items()},
}
with open('modelos/metadatos.json','w') as fh: json.dump(meta, fh, indent=2, ensure_ascii=False)
rating_factors(modelo_frecuencia).to_csv('modelos/rf_frecuencia.csv')
rating_factors(modelo_severidad).to_csv('modelos/rf_severidad.csv')
rating_factors(modelo_agregado).to_csv('modelos/rf_agregado.csv')

print('Guardados en modelos/ (ligero):')
for _f in sorted(os.listdir('modelos')):
    kb = os.path.getsize('modelos/' + _f) / 1024
    print(f'   {_f}  ({kb:.1f} KB)')
print('Reconstrucción en la Sesión 3:  from cargar_modelos import cargar_modelos')

Guardados en modelos/ (ligero):
   metadatos.json  (1.3 KB)
   rf_agregado.csv  (2.5 KB)
   rf_frecuencia.csv  (3.5 KB)
   rf_severidad.csv  (3.1 KB)
Reconstrucción en la Sesión 3:  from cargar_modelos import cargar_modelos


### Python vs R — las tres distribuciones, y el detalle de Tweedie

**Python**
```python
# Severidad Gamma
smf.glm('severidad ~ ...', sev, family=sm.families.Gamma(links.Log()),
        freq_weights=sev.num_siniestros).fit()
# Tweedie: p por barrido de D² (sklearn) + GLM (statsmodels)
TweedieRegressor(power=p, link='log').fit(X, y, sample_weight=e)
smf.glm('pure_premium ~ ...', family=sm.families.Tweedie(var_power=p))
```

**R**
```r
glm(severidad ~ ..., family = Gamma(link="log"), weights = num_siniestros)   # Gamma
library(statmod); library(tweedie)
p <- tweedie.profile(pp ~ ..., p.vec = seq(1.1,1.9,0.1), do.plot=TRUE)$p.max # p por MLE
glm(pp ~ ..., family = tweedie(var.power = p, link.power = 0), weights = expo)
```

**La diferencia que enseña:** `sklearn` estima $p$ por un **barrido de D²** (rápido, aproximado);
`tweedie.profile` de R lo estima por **máxima verosimilitud** perfilada (más costoso, más
preciso). Misma idea, distinto rigor numérico — bueno tenerlo presente al reportar $p$.

| Concepto | Python | R |
|---|---|---|
| Gamma | `families.Gamma(links.Log())` | `Gamma(link="log")` |
| pseudo R² | construido a mano | `DescTools::PseudoR2` |
| Tweedie p | `TweedieRegressor` (D²) | `tweedie.profile` (MLE) |
| Stepwise | loop por AIC | `step()` |

---
## 🎯 Ejercicio Integrador — Elige y guarda TU mejor tarifa

1. Corre **stepwise** para frecuencia y severidad; documenta las variables elegidas.
2. Ajusta **Tweedie** con su $p$ óptimo; compara la prima pura total vs Freq × Sev.
3. Compara los candidatos por **AIC, BIC y pseudo R²** y justifica tu elección.
4. **Guarda** tus tres modelos en `modelos/` (ya hay funciones arriba).
5. En 3–4 líneas: ¿Freq × Sev o Tweedie para nota técnica CNSF, y por qué?

*Solución en `solucion_integrador()`.*

In [21]:
# Scaffold — reutiliza forward_stepwise, pseudo_r2 y rating_factors de arriba.
# El objetivo es dejar modelos/ listo para la Sesión 3.

---
## Cierre — lo que llevas y lo que sigue

Hoy completaste la prima pura (**severidad Gamma**), aprendiste a domar los **large losses**,
conociste el atajo **Tweedie**, y —lo central— a **elegir el mejor modelo** con AIC/BIC/pseudo R²
y **stepwise automatizado**. Y dejaste **tres modelos guardados** en `modelos/`.

**Sesión 3 (28 ago):** los ponemos a trabajar — **visualización de efectos** (con plotly),
**validación out-of-sample** (Gini, calibración, lift), **deducible y suma asegurada**,
**análisis de sensibilidad** y **simulación de tarifa**. Y al cerrar el tema: el **MCP** que
interpreta las salidas del GLM por ti.